# Scrape Google Trends for Your Own Topic - Lane A (the real experience: you prompt, the agent builds)

**SISMID 2026 - Day 1, 3:30 session.** You drive a coding agent (Codex, Claude Code, or
Antigravity CLI) to pull Google Trends for **your own** disease and place, and keep the
judgment. For each step: read the goal, paste the **prompt**, run the code, apply the check.

> Each prompt produces roughly the matching cell in the **Lane B** notebook. Not set up
> with an agent yet? Use Lane B. Default example topic: flu in the US.


## Step 0: set up a robust fetch, and pick your topic

> *Using pytrends, write `gt_fetch(kw_list, timeframe, geo)` that returns a tidy DataFrame*
> *(date + one column per entry, underscored names) and accepts literal phrases, additive*
> *"a + b" queries, or topic mids. Build TrendReq with retries and a small random pause;*
> *on a 429 wait ~10s and retry a few times; return None (not raise) if it still fails.*
> *Also write `topic_mid(phrase)` that resolves a phrase to a Google Trends topic entity*
> *via pytrends suggestions, and a loader for the cached example CSV in data/. Then set*
> *two variables I will edit: MY_TERMS (my disease's query phrases) and MY_GEO.*


In [ ]:
from pytrends.request import TrendReq   # pip install pytrends (in requirements.txt)
import pandas as pd, os, time, random

# ===== EDIT THESE TWO LINES for your own topic =====
MY_TERMS = ['dengue', 'mosquito', 'space spraying']   # a handful of query phrases, max 5
MY_GEO   = 'US'                            # e.g. 'US', 'MX', 'US-GA', 'BR'
# ===================================================

# a cached example (flu, US) so this notebook still runs if the live pull is blocked
CACHE_PATHS = [
    '../data/google_trends_flu_us_cached.csv',
    'data/google_trends_flu_us_cached.csv',
    './google_trends_flu_us_cached.csv',
]

def _norm(c):
    return c.strip().replace(' ', '_')

def gt_fetch(kw_list, timeframe, geo, tries=4):
    """Google Trends interest-over-time for a handful of terms/topics (max 5).
    kw_list entries can be literal phrases, additive queries like 'flu + gripe',
    or topic mids like '/m/0cycc'. Returns a tidy DataFrame (date + one column
    per entry), or None if Google keeps refusing. A first 429 is normal even from
    a Codespace (Azure IP); we wait and retry. The small random pause staggers the
    room so we do not all hit Google at once."""
    time.sleep(random.uniform(0, 3))   # stagger: do NOT count '3-2-1-everyone run'
    for attempt in range(tries):
        try:
            pt = TrendReq(hl='en-US', tz=360, retries=2, backoff_factor=0.5)
            pt.build_payload(kw_list, timeframe=timeframe, geo=geo)
            df = pt.interest_over_time()
            if df.empty:
                raise RuntimeError('empty frame (rate-limited)')
            df = df.drop(columns=[c for c in ['isPartial'] if c in df.columns]).reset_index()
            return df.rename(columns={c: _norm(c) for c in df.columns})
        except Exception as e:
            rate_limited = ('429' in str(e) or 'empty frame' in str(e)
                            or 'toomany' in type(e).__name__.lower())
            if rate_limited and attempt < tries - 1:
                wait = 10 * (attempt + 1)
                print(f'Rate-limited (attempt {attempt+1}/{tries}); waiting {wait}s and retrying...')
                time.sleep(wait)
                continue
            print(f'Live Google Trends pull failed ({type(e).__name__}): {e}')
            return None
    return None

def topic_mid(phrase):
    """Resolve a phrase to a Google Trends TOPIC entity (a Knowledge Graph 'mid'
    like '/m/0cycc') via pytrends suggestions. Returns (mid, title, type); falls
    back to the literal phrase if lookup fails."""
    try:
        s = TrendReq(hl='en-US', tz=360).suggestions(phrase)
        if s:
            return s[0]['mid'], s[0]['title'], s[0]['type']
    except Exception as e:
        print('suggestions lookup failed:', e)
    return phrase, phrase, 'raw term'

def load_cache():
    for p in CACHE_PATHS:
        if os.path.exists(p):
            print(f'Using cached example (flu, US): {p}')
            return pd.read_csv(p, parse_dates=['date'])
    raise FileNotFoundError('Cached example not found; check the data/ folder.')


## Step 1: pull your terms, live

> *Use gt_fetch to pull MY_TERMS for MY_GEO over the past 5 years (today 5-y). If it*
> *returns None, fall back to the cached example. Plot the series and print the last date.*

**Your check:** does the last point land on the current week, and do peaks fall in a
plausible season for your disease and place?


In [2]:
# Agent's live pull (with cache fallback):


## Step 2: term vs topic

> *Compare three ways to get the signal: (1) the individual terms, (2) one combined*
> *additive query joining the phrases with ' + ' (a topic-like OR), and (3) the real*
> *Google Trends topic entity, resolved with topic_mid and pulled by its mid. Report the*
> *percent of zeros for each, and overlay them on one plot. Which has the fewest zeros and*
> *the cleanest seasonal signal?*

**Why it matters:** a raw term misses synonyms, spellings, and languages; a topic folds
them in, with more volume and fewer zeros.


In [3]:
# Agent's term-vs-topic comparison:


## Step 3: expose the instability

> *Pull the first term twice with the same window and geo, merge on date, and report*
> *whether the two pulls are identical and their mean absolute difference.*


In [4]:
# Agent's twice-pulled comparison:


## Step 4: sanity-check and save

> *Report the geo, date range, row count, missing values per column, and which terms vary.*
> *Then save the tidy table to my_topic_search.csv and confirm it was written.*


In [5]:
# Agent's sanity-check + save:


## Reflection

- You described the outcome; the agent wrote the pytrends plumbing, retries and all.
- You saw the **term vs topic** difference first-hand, and know when to prefer a combined
  or topic signal.
- **If the whole room keeps getting blocked:** route through a proxy or VPN (see the
  slides and `proxy-setup.md`).
- **Day 2:** the same loop for Wikipedia, wastewater, mobility, weather, and news.
